# Module 13 Lab - Building ML Pipelines**Objective:** To understand the importance of `scikit-learn` **Pipelines** for creating robust, reproducible, and professional machine learning workflows.**In this lab, you will refactor code from a previous lab into a clean, professional `Pipeline` object.**

## Part 1: Why Use Pipelines?**Concept:** As you've seen, a typical ML workflow involves multiple steps: loading data, cleaning it, splitting it, preprocessing features (scaling, encoding), and finally, training a model. Managing all these steps separately can be messy and error-prone.**Data Leakage:** A major risk of manual preprocessing is **data leakage**. This happens when information from the test set accidentally "leaks" into the training process. For example, if you calculate the mean for scaling using the *entire* dataset before splitting, the model has already "seen" the test data, leading to overly optimistic performance estimates.**A `scikit-learn` Pipeline solves these problems by:**1.  **Encapsulating** all workflow steps into a single object.2.  **Preventing Data Leakage:** It ensures that preprocessing steps are fitted *only* on the training data during cross-validation or when calling `.fit()`.3.  **Improving Reproducibility:** The entire workflow is saved as one object, making it easy to reuse and deploy.

## Part 2: The "Manual" Way (What We Did Before)Let's revisit the Titanic dataset and the steps we took to prepare the data and train a model. This code should look familiar. Notice how many separate objects and steps there are.

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Basic feature engineering and cleaning
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df.drop('Cabin', axis=1, inplace=True)

X = df.drop(['Survived', 'Name', 'Ticket', 'PassengerId'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify feature types
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Manual Preprocessing
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_num = scaler.transform(X_test[numeric_features]) # Note: using .transform() here!

encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded_cat = encoder.fit_transform(X_train[categorical_features])
X_test_encoded_cat = encoder.transform(X_test[categorical_features])

# Combine preprocessed features
X_train_processed = np.hstack((X_train_scaled_num, X_train_encoded_cat.toarray()))
X_test_processed = np.hstack((X_test_scaled_num, X_test_encoded_cat.toarray()))

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)

print(f"Accuracy (Manual Method): {accuracy_score(y_test, y_pred):.2%}")

/tmp/ipykernel_14658/796389952.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_14658/796389952.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

Accuracy (Manual Method): 82.68%


## Part 3: The "Pipeline" WayNow, let's do the exact same thing but encapsulate all the preprocessing steps into a single `Pipeline`.**Your Task:** Use `make_pipeline` and `make_column_transformer` to build a complete workflow. This is the modern, professional way to build models in `scikit-learn`.

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
# Reload the data to start fresh
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# --- ENTER YOUR CODE HERE ---
# 1. Create a pipeline for numeric features
#    This pipeline will first impute missing 'Age' values with the median, then scale the features.
# numeric_transformer = make_pipeline(
#     SimpleImputer(strategy='median'),
#     StandardScaler()
# )
# 2. Create a pipeline for categorical features
#    This pipeline will first impute missing 'Embarked' values with the most frequent value, then one-hot encode.
# categorical_transformer = make_pipeline(
#     SimpleImputer(strategy='most_frequent'),
#     OneHotEncoder(handle_unknown='ignore')
# )
# 3. Use ColumnTransformer to apply different transformers to different columns
# preprocessor = make_column_transformer(
#     (numeric_transformer, ['Age', 'Fare', 'SibSp', 'Parch']),
#     (categorical_transformer, ['Pclass', 'Sex', 'Embarked'])
# )
# 4. Create the final, full pipeline
#    This chains the preprocessor and the final model together.
# final_pipeline = make_pipeline(
#     preprocessor,
#     RandomForestClassifier(random_state=42)
# )
# 5. Fit and evaluate the entire pipeline in one step!
# final_pipeline.fit(X_train, y_train)
# y_pred_pipeline = final_pipeline.predict(X_test)
# print(f"Accuracy (Pipeline Method): {accuracy_score(y_test, y_pred_pipeline):.2%}")

## 📝 Reflective Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

1.  **Code Comparison:** Look at the "Manual Way" versus the "Pipeline Way". What are the three biggest advantages you see in using the Pipeline approach?
2.  **Data Leakage Explained:** In the manual code, we used `scaler.fit_transform()` on the training data but only `scaler.transform()` on the test data. Why was this distinction crucial? How does the Pipeline automatically handle this for you?
3.  **Extending the Pipeline:** Imagine you wanted to add a PCA step to reduce dimensionality *after* scaling and encoding but *before* the RandomForestClassifier. How would you modify your `final_pipeline` object to include this step? (You don't need to write the full code, just describe where you would add `PCA()`.)
4.  **Real-World Value:** You are handing your model over to another team to deploy into a web application. Why is giving them the single `final_pipeline` object much safer and more reliable than giving them the 5 separate objects (`scaler`, `encoder`, `model`, etc.) from the manual approach?

**[ENTER YOUR ANSWERS HERE]**

1.  **Code Comparison Advantages of Pipelines:**
    *   **Streamlined Workflow:** Pipelines beautifully encapsulate all preprocessing and modeling steps into a single, cohesive object. This makes the code much cleaner, easier to read, and simpler to manage compared to handling each step individually.
    *   **Data Leakage Prevention:** This is a critical advantage! Pipelines inherently ensure that any transformations (like scaling or encoding) are `fit` only on the training data and then `transformed` consistently across both training and test sets. This elegantly prevents data from the test set from inadvertently influencing the training process.
    *   **Enhanced Reproducibility & Deployment:** With a single pipeline object, the entire machine learning workflow is precisely defined and executed in a consistent order every time. This greatly simplifies reproducing results and makes deployment in production environments much more robust and less prone to errors.

2.  **Data Leakage Explained:**
    *   **Crucial Distinction:** The distinction between using `scaler.fit_transform()` on training data and `scaler.transform()` on test data in the manual approach is vital to prevent **data leakage**. When `fit()` is called, the scaler learns parameters (e.g., mean and standard deviation for `StandardScaler` or unique categories for `OneHotEncoder`) from the *training data only*. If we were to use `fit_transform()` on the test data, the scaler would learn parameters from data it shouldn't have seen during training, leading to an overly optimistic and unrealistic evaluation of the model's performance on new, unseen data.
    *   **How Pipelines Handle This:** The `Pipeline` conveniently handles this automatically. When you call `.fit(X_train, y_train)` on a pipeline, it internally calls `fit_transform()` on the training data for all intermediate steps (like `SimpleImputer` and `StandardScaler`), and then `fit()` for the final estimator. When you subsequently call `.predict(X_test)` or `.transform(X_test)`, the pipeline ensures that only the `transform()` method of the intermediate steps is applied to the test data, using the parameters *already learned* from the training phase. It's a clean and robust solution!

3.  **Extending the Pipeline with PCA:**
    *   To add a PCA step, I would insert `PCA()` as a new step directly into the `make_pipeline` call for the `final_pipeline`, placing it *after* the `preprocessor` step and *before* the `RandomForestClassifier` step. The pipeline structure would then automatically apply PCA to the features after they have been scaled and encoded.

4.  **Real-World Value for Deployment:**
    *   Providing a single `final_pipeline` object to another team for deployment offers immense value. It acts as a self-contained, end-to-end solution for the model. Instead of needing to coordinate 5 separate objects (`scaler`, `encoder`, `model`, etc.) and ensure they are applied in the correct sequence with the right configurations, the deployment team simply needs to load and use one `final_pipeline` object. This significantly reduces the risk of human error, simplifies the deployment process, ensures consistency across different environments, and makes the model much more maintainable and reliable in a production setting. It's a single source of truth for the entire predictive process.